# Clean-hard targeted training vs random control

RQ1: does up-weighting hard rows with a clear label beat up-weighting the same number of random rows?
RQ2: does the answer change when the FinBERT embeddings are fine-tuned instead of frozen?

Every run retrains the FinBERT CNN from scratch with the baseline recipe; only the loss weights change.

| arm | rows weighted K (all others 1) |
|---|---|
| `no_weight` | none, the reference |
| `clean_hard` | 103 hard + all-agree rows |
| `random` | 103 random train rows (new draw per seed) |
| `clean_random` | 103 all-agree, non-hard rows (new draw per seed) |
| `variant` | clean-hard rows get K, the 261 hard rows without agreement get 0.5 |

Row sets come from `results/rq1_rq2/row_sets.csv`, scores are on **val** only. K is chosen here; test is run once at the end.
Every run goes to `results/rq1_rq2/runs.csv` (re-running a run replaces it) and its val probabilities to `results/rq1_rq2/probs/` for the taxonomy.

## Imports

In [1]:
import os
os.environ["HF_HOME"] = os.path.abspath("../.cache/huggingface")
os.environ["HF_HUB_CACHE"] = os.path.abspath("../.cache/huggingface/hub")
import sys

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from tensorflow import keras

sys.path.append("..")
from utils.finbert_cnn import SEEDS, load_finbert_inputs, train_baseline
from utils.preprocessing import encode_labels

## Helpers

In [ ]:
# arms we compare. hard = unsure (top probability in the 0.4-0.6 band)
ARMS = ["no_weight", "clean_hard", "random", "clean_random", "variant"]
AMBIGUOUS_WEIGHT = 0.5
RUN_KEY = ["arm", "k", "embedding_config", "seed"]


def build_row_sets(band_train, seeds):
    # which train rows get the extra weight in each arm
    hard = band_train["is_hard"].values
    agree = band_train["all_agree"].values

    clean_hard = np.where(hard & agree)[0]
    ambiguous_hard = np.where(hard & ~agree)[0]
    clean_easy = np.where(~hard & agree)[0]

    rows = []
    for seed in seeds:
        rng = np.random.default_rng(seed)
        sets = {}
        sets["clean_hard"] = clean_hard
        sets["ambiguous_hard"] = ambiguous_hard
        sets["random"] = rng.choice(len(band_train), len(clean_hard), replace=False)
        sets["clean_random"] = rng.choice(clean_easy, len(clean_hard), replace=False)

        for name in sets:
            for i in sorted(sets[name]):
                rows.append({"row_set": name, "seed": seed, "train_idx": i})
    return pd.DataFrame(rows)


def rows_for(row_sets, name, seed):
    picked = row_sets[(row_sets["row_set"] == name) & (row_sets["seed"] == seed)]
    return picked["train_idx"].values


def make_weights(row_sets, arm, k, seed, n_train):
    if arm not in ARMS:
        raise ValueError("unknown arm: " + arm)

    weights = np.ones(n_train)
    if arm == "no_weight":
        return weights
    if arm == "variant":
        weights[rows_for(row_sets, "ambiguous_hard", seed)] = AMBIGUOUS_WEIGHT
        weights[rows_for(row_sets, "clean_hard", seed)] = k
    else:
        weights[rows_for(row_sets, arm, seed)] = k
    return weights


def score_subsets(y_true, probs, band):
    y_true = np.array(y_true)
    pred = np.array(probs).argmax(axis=1)
    hard = band["is_hard"].values
    agree = band["all_agree"].values

    subsets = {
        "overall": np.ones(len(y_true), dtype=bool),
        "hard": hard,
        "nonhard": ~hard,
        "allagree": agree,
        "not_allagree": ~agree,
    }
    scores = {}
    for name in subsets:
        mask = subsets[name]
        scores["accuracy_" + name] = accuracy_score(y_true[mask], pred[mask])
        scores["f1_" + name] = f1_score(y_true[mask], pred[mask], average="macro", zero_division=0)

    per_class = f1_score(y_true, pred, labels=[0, 1, 2], average=None, zero_division=0)
    scores["f1_negative"] = per_class[0]
    scores["f1_neutral"] = per_class[1]
    scores["f1_positive"] = per_class[2]
    return scores


def run_arm(arm, k, seed, data, row_sets, trainable=False, epochs=20):
    # trains one model from scratch with the weights of this arm, scores it on val

    weights = make_weights(row_sets, arm, k, seed, len(data["y_train"]))
    keras.utils.set_random_seed(seed)
    model = train_baseline(data["embedding_matrix"], data["X_train"], data["y_train"],
                           data["X_val"], data["y_val"], trainable=trainable, verbose=0,
                           epochs=epochs, sample_weight=weights)
    probs = model.predict(data["X_val"], batch_size=32, verbose=0)

    if arm == "no_weight":
        k = 1
    if trainable:
        embedding_config = "finbert_finetuned"
    else:
        embedding_config = "finbert_frozen"

    row = {"arm": arm, "k": k, "embedding_config": embedding_config, "seed": seed}
    row.update(score_subsets(data["y_val"], probs, data["band_val"]))
    return row, probs


def save_runs(path, new_rows):
    # if a run is done again, the new result replaces the old one
    if os.path.exists(path):
        old = pd.read_csv(path)
        new_rows = pd.concat([old, new_rows], ignore_index=True)
    new_rows = new_rows.drop_duplicates(subset=RUN_KEY, keep="last")
    new_rows = new_rows.sort_values(RUN_KEY).reset_index(drop=True)
    new_rows.to_csv(path, index=False)
    return new_rows


## Data

In [ ]:
train_df = pd.read_csv("../data/train.csv")
val_df = pd.read_csv("../data/val.csv")
y_train = encode_labels(train_df.label)
y_val = encode_labels(val_df.label)
X_train, X_val, embedding_matrix = load_finbert_inputs(train_df, val_df)

row_sets = pd.read_csv("../results/rq1_rq2/row_sets.csv")
data = dict(embedding_matrix=embedding_matrix, X_train=X_train, y_train=y_train, X_val=X_val, y_val=y_val,
            band_val=pd.read_csv("../results/hard_samples/band_val.csv"))

RUNS_PATH = "../results/rq1_rq2/runs.csv"
PROBS_DIR = "../results/rq1_rq2/probs"
os.makedirs(PROBS_DIR, exist_ok=True)
KS = [3, 5, 10]
print(row_sets.groupby("row_set").train_idx.count() // len(SEEDS))

In [3]:
def run_seeds(arm, k, trainable=False, epochs=20):
    rows = []
    for seed in SEEDS:
        row, probs = run_arm(arm, k, seed, data, row_sets, trainable=trainable, epochs=epochs)
        rows.append(row)
        name = f"{arm}_k{row['k']}_{row['embedding_config']}_s{seed}"
        np.save(f"{PROBS_DIR}/{name}.npy", probs)
        print(f"{arm} k={row['k']} seed={seed}: f1={row['f1_overall']:.3f}  hard={row['f1_hard']:.3f}")
    return save_runs(RUNS_PATH, pd.DataFrame(rows))


## 1. No-weight reference (run this first)

Check: every seed should land around 0.74-0.77. A seed at about 0.248 means it got stuck predicting neutral; stop and fix that before running the other arms.

In [4]:
runs = run_seeds("no_weight", 1)

 - val_macro_f1: 0.3184
 - val_macro_f1: 0.4151
 - val_macro_f1: 0.4368
 - val_macro_f1: 0.5897
 - val_macro_f1: 0.6485
 - val_macro_f1: 0.6796
 - val_macro_f1: 0.7090
 - val_macro_f1: 0.7188
 - val_macro_f1: 0.7110
 - val_macro_f1: 0.7303
 - val_macro_f1: 0.7102
 - val_macro_f1: 0.7318
 - val_macro_f1: 0.7275
 - val_macro_f1: 0.7436
 - val_macro_f1: 0.7451
 - val_macro_f1: 0.7605
 - val_macro_f1: 0.7653
 - val_macro_f1: 0.7455
 - val_macro_f1: 0.7468
 - val_macro_f1: 0.7354
no_weight k=1 seed=42: f1=0.765  hard=0.606
 - val_macro_f1: 0.3505
 - val_macro_f1: 0.4550
 - val_macro_f1: 0.6215
 - val_macro_f1: 0.6913
 - val_macro_f1: 0.7415
 - val_macro_f1: 0.7549
 - val_macro_f1: 0.7519
 - val_macro_f1: 0.7570
 - val_macro_f1: 0.7603
 - val_macro_f1: 0.7525
 - val_macro_f1: 0.7608
 - val_macro_f1: 0.7498
 - val_macro_f1: 0.7576
 - val_macro_f1: 0.7508
 - val_macro_f1: 0.7296
 - val_macro_f1: 0.7423
no_weight k=1 seed=43: f1=0.761  hard=0.617
 - val_macro_f1: 0.3246
 - val_macro_f1: 0.4144


## 2. Targeted arms (frozen embeddings)

In [5]:
for k in KS:
    runs = run_seeds("clean_hard", k)

 - val_macro_f1: 0.3875
 - val_macro_f1: 0.4194
 - val_macro_f1: 0.4652
 - val_macro_f1: 0.6428
 - val_macro_f1: 0.6661
 - val_macro_f1: 0.6761
 - val_macro_f1: 0.7226
 - val_macro_f1: 0.7182
 - val_macro_f1: 0.7428
 - val_macro_f1: 0.7184
 - val_macro_f1: 0.7336
 - val_macro_f1: 0.7353
 - val_macro_f1: 0.7295
 - val_macro_f1: 0.7653
 - val_macro_f1: 0.7545
 - val_macro_f1: 0.7476
 - val_macro_f1: 0.7524
 - val_macro_f1: 0.7522
 - val_macro_f1: 0.7329
clean_hard k=3 seed=42: f1=0.765  hard=0.571
 - val_macro_f1: 0.2950
 - val_macro_f1: 0.4653
 - val_macro_f1: 0.6276
 - val_macro_f1: 0.7039
 - val_macro_f1: 0.7368
 - val_macro_f1: 0.7485
 - val_macro_f1: 0.7626
 - val_macro_f1: 0.7634
 - val_macro_f1: 0.7659
 - val_macro_f1: 0.7573
 - val_macro_f1: 0.7686
 - val_macro_f1: 0.7375
 - val_macro_f1: 0.7386
 - val_macro_f1: 0.7409
 - val_macro_f1: 0.7217
 - val_macro_f1: 0.7010
clean_hard k=3 seed=43: f1=0.769  hard=0.587
 - val_macro_f1: 0.3154
 - val_macro_f1: 0.4474
 - val_macro_f1: 0.488

In [6]:
for k in KS:
    runs = run_seeds("variant", k)

 - val_macro_f1: 0.3859
 - val_macro_f1: 0.4130
 - val_macro_f1: 0.4539
 - val_macro_f1: 0.5859
 - val_macro_f1: 0.6447
 - val_macro_f1: 0.6678
 - val_macro_f1: 0.7091
 - val_macro_f1: 0.7078
 - val_macro_f1: 0.7119
 - val_macro_f1: 0.7224
 - val_macro_f1: 0.7249
 - val_macro_f1: 0.7248
 - val_macro_f1: 0.7318
 - val_macro_f1: 0.7240
 - val_macro_f1: 0.7293
 - val_macro_f1: 0.7424
 - val_macro_f1: 0.7330
 - val_macro_f1: 0.7374
 - val_macro_f1: 0.7410
 - val_macro_f1: 0.7345
variant k=3 seed=42: f1=0.742  hard=0.482
 - val_macro_f1: 0.2973
 - val_macro_f1: 0.4561
 - val_macro_f1: 0.5698
 - val_macro_f1: 0.6557
 - val_macro_f1: 0.7413
 - val_macro_f1: 0.7406
 - val_macro_f1: 0.7577
 - val_macro_f1: 0.7687
 - val_macro_f1: 0.7599
 - val_macro_f1: 0.7667
 - val_macro_f1: 0.7540
 - val_macro_f1: 0.7297
 - val_macro_f1: 0.7408
variant k=3 seed=43: f1=0.769  hard=0.569
 - val_macro_f1: 0.3603
 - val_macro_f1: 0.4439
 - val_macro_f1: 0.5008
 - val_macro_f1: 0.5793
 - val_macro_f1: 0.6397
 - v

## 3. Controls (frozen embeddings)

In [7]:
for k in KS:
    runs = run_seeds("random", k)

 - val_macro_f1: 0.3813
 - val_macro_f1: 0.4151
 - val_macro_f1: 0.4477
 - val_macro_f1: 0.6264
 - val_macro_f1: 0.6616
 - val_macro_f1: 0.6846
 - val_macro_f1: 0.6881
 - val_macro_f1: 0.7232
 - val_macro_f1: 0.7217
 - val_macro_f1: 0.7157
 - val_macro_f1: 0.7315
 - val_macro_f1: 0.7434
 - val_macro_f1: 0.7243
 - val_macro_f1: 0.7351
 - val_macro_f1: 0.7383
 - val_macro_f1: 0.7466
 - val_macro_f1: 0.7364
 - val_macro_f1: 0.7347
 - val_macro_f1: 0.7462
 - val_macro_f1: 0.7198
random k=3 seed=42: f1=0.747  hard=0.485
 - val_macro_f1: 0.3350
 - val_macro_f1: 0.4917
 - val_macro_f1: 0.6499
 - val_macro_f1: 0.6803
 - val_macro_f1: 0.7198
 - val_macro_f1: 0.7413
 - val_macro_f1: 0.7388
 - val_macro_f1: 0.7471
 - val_macro_f1: 0.7582
 - val_macro_f1: 0.7431
 - val_macro_f1: 0.7486
 - val_macro_f1: 0.7195
 - val_macro_f1: 0.7239
 - val_macro_f1: 0.7367
random k=3 seed=43: f1=0.758  hard=0.622
 - val_macro_f1: 0.3306
 - val_macro_f1: 0.4159
 - val_macro_f1: 0.4419
 - val_macro_f1: 0.4974
 - val

In [8]:
for k in KS:
    runs = run_seeds("clean_random", k)

 - val_macro_f1: 0.3789
 - val_macro_f1: 0.4167
 - val_macro_f1: 0.4383
 - val_macro_f1: 0.5355
 - val_macro_f1: 0.6523
 - val_macro_f1: 0.6658
 - val_macro_f1: 0.7104
 - val_macro_f1: 0.7206
 - val_macro_f1: 0.7105
 - val_macro_f1: 0.7178
 - val_macro_f1: 0.7286
 - val_macro_f1: 0.7260
 - val_macro_f1: 0.7107
 - val_macro_f1: 0.7264
 - val_macro_f1: 0.7353
 - val_macro_f1: 0.7522
 - val_macro_f1: 0.7475
 - val_macro_f1: 0.7393
 - val_macro_f1: 0.7366
 - val_macro_f1: 0.7351
clean_random k=3 seed=42: f1=0.752  hard=0.566
 - val_macro_f1: 0.3362
 - val_macro_f1: 0.4335
 - val_macro_f1: 0.6169
 - val_macro_f1: 0.6845
 - val_macro_f1: 0.7218
 - val_macro_f1: 0.7475
 - val_macro_f1: 0.7510
 - val_macro_f1: 0.7534
 - val_macro_f1: 0.7607
 - val_macro_f1: 0.7514
 - val_macro_f1: 0.7452
 - val_macro_f1: 0.7338
 - val_macro_f1: 0.7329
 - val_macro_f1: 0.7235
clean_random k=3 seed=43: f1=0.761  hard=0.530
 - val_macro_f1: 0.2754
 - val_macro_f1: 0.4020
 - val_macro_f1: 0.4873
 - val_macro_f1: 0

## 4. Compare on val and choose K

Mean and spread over the seeds, and the difference to the no-weight reference. Pick K where clean-hard minus its controls is largest **compared with the spread**; a difference smaller than the std is not an effect.

In [ ]:
runs = pd.read_csv(RUNS_PATH)
cols = ["f1_overall", "f1_hard", "f1_nonhard", "f1_allagree", "f1_not_allagree"]

summary = runs.groupby(["embedding_config", "arm", "k"])[cols].agg(["mean", "std"]).round(3)
summary

In [ ]:
frozen = runs[runs.embedding_config == "finbert_frozen"]
reference = frozen[frozen.arm == "no_weight"][cols].mean()

# how much each arm gains over no_weight (mean over seeds)
gain = frozen[frozen.arm != "no_weight"].groupby(["arm", "k"])[cols].mean() - reference
gain.round(3)


In [ ]:
# targeted minus its control, seed by seed (both arms use the same seeds)
pairs = [("clean_hard", "random"), ("clean_hard", "clean_random")]

for target, control in pairs:
    for k in sorted(frozen[frozen.arm == target].k.unique()):
        a = frozen[(frozen.arm == target) & (frozen.k == k)].sort_values("seed").f1_overall.values
        b = frozen[(frozen.arm == control) & (frozen.k == k)].sort_values("seed").f1_overall.values
        if len(a) > 0 and len(a) == len(b):
            diff = a - b
            print(f"{target} - {control}, k={k}: mean {diff.mean():.3f}, std {diff.std(ddof=1):.3f}, "
                  f"min {diff.min():.3f}, max {diff.max():.3f}")


## 5. RQ2: fine-tuned embeddings at the chosen K

Run clean-hard and its random control at the chosen K. Fine-tuned embeddings learn slowly at lr 5e-5, so they get 80 epochs.

In [17]:
TARGET, CONTROL = "clean_hard", "random"
CHOSEN_K = 5    # fill in after section 4
assert CHOSEN_K in KS, "choose K in section 4 first"

for arm, k in [("no_weight", 1), (TARGET, CHOSEN_K), (CONTROL, CHOSEN_K)]:
    runs = run_seeds(arm, k, trainable=True, epochs=80)


 - val_macro_f1: 0.3910
 - val_macro_f1: 0.5998
 - val_macro_f1: 0.7122
 - val_macro_f1: 0.7239
 - val_macro_f1: 0.6888
 - val_macro_f1: 0.6946
 - val_macro_f1: 0.6908
 - val_macro_f1: 0.6889
 - val_macro_f1: 0.6958
no_weight k=1 seed=42: f1=0.724  hard=0.426
 - val_macro_f1: 0.4962
 - val_macro_f1: 0.6575
 - val_macro_f1: 0.6949
 - val_macro_f1: 0.7094
 - val_macro_f1: 0.7057
 - val_macro_f1: 0.7035
 - val_macro_f1: 0.7204
 - val_macro_f1: 0.7085
 - val_macro_f1: 0.7131
 - val_macro_f1: 0.7132
 - val_macro_f1: 0.7117
 - val_macro_f1: 0.7079
no_weight k=1 seed=43: f1=0.720  hard=0.465
 - val_macro_f1: 0.4460
 - val_macro_f1: 0.6229
 - val_macro_f1: 0.6872
 - val_macro_f1: 0.6923
 - val_macro_f1: 0.7022
 - val_macro_f1: 0.6999
 - val_macro_f1: 0.6920
 - val_macro_f1: 0.6912
 - val_macro_f1: 0.6717
 - val_macro_f1: 0.6711
no_weight k=1 seed=44: f1=0.702  hard=0.237
 - val_macro_f1: 0.5087
 - val_macro_f1: 0.6674
 - val_macro_f1: 0.6937
 - val_macro_f1: 0.6963
 - val_macro_f1: 0.7043
 - v